# Ex 1.6 Intro to NLP and Network Analysis 

### 1. Install libraries 

In [18]:
# Spacy doesn't work in Python 3.13 so I've had to make a new virtual environment
# running Python 3.11 and installed spacy there.

In [8]:
!conda install pandas -y

Jupyter detected...
2 channel Terms of Service accepted
Channels:
 - defaults
Platform: osx-64
Solving environment: done

## Package Plan ##

  environment location: /opt/anaconda3/envs/nlp_env

  added / updated specs:
    - pandas


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    bottleneck-1.4.2           |  py311h9b7fc35_0         153 KB
    numexpr-2.11.0             |  py311h20bc273_0         208 KB
    numpy-2.0.1                |  py311h13f252e_1          12 KB
    numpy-base-2.0.1           |  py311ha745260_1         8.4 MB
    pandas-2.3.1               |  py311hebe84f7_0        15.0 MB
    pytz-2025.2                |  py311hecd8cb5_0         239 KB
    ------------------------------------------------------------
                                           Total:        24.0 MB

The following NEW packages will be INSTALLED:

  blas               pkgs/main/osx-64::blas-1.0-openb

In [10]:
!conda install networkx -y

Jupyter detected...
2 channel Terms of Service accepted
Channels:
 - defaults
Platform: osx-64
Solving environment: done

## Package Plan ##

  environment location: /opt/anaconda3/envs/nlp_env

  added / updated specs:
    - networkx


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    networkx-3.4.2             |  py311hecd8cb5_0         3.1 MB
    ------------------------------------------------------------
                                           Total:         3.1 MB

The following NEW packages will be INSTALLED:

  networkx           pkgs/main/osx-64::networkx-3.4.2-py311hecd8cb5_0 



                                                                                
Preparing transaction: done
Verifying transaction: done
Executing transaction: done


In [12]:
!conda install matplotlib -y

Jupyter detected...
2 channel Terms of Service accepted
Channels:
 - defaults
Platform: osx-64
Solving environment: done

## Package Plan ##

  environment location: /opt/anaconda3/envs/nlp_env

  added / updated specs:
    - matplotlib


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    contourpy-1.3.1            |  py311h1962661_0         270 KB
    fonttools-4.55.3           |  py311h46256e1_0         2.9 MB
    kiwisolver-1.4.8           |  py311h6d0c2b6_0          60 KB
    matplotlib-3.10.0          |  py311hecd8cb5_0           8 KB
    matplotlib-base-3.10.0     |  py311h919b35b_0         8.3 MB
    pillow-11.3.0              |  py311h25d8182_0         968 KB
    pyparsing-3.2.0            |  py311hecd8cb5_0         480 KB
    unicodedata2-15.1.0        |  py311h46256e1_1         517 KB
    ------------------------------------------------------------
                                

In [14]:
!conda install scipy -y

Jupyter detected...
2 channel Terms of Service accepted
Channels:
 - defaults
Platform: osx-64
Solving environment: done

## Package Plan ##

  environment location: /opt/anaconda3/envs/nlp_env

  added / updated specs:
    - scipy


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    scipy-1.16.0               |  py311hcd1e4c7_0        23.7 MB
    ------------------------------------------------------------
                                           Total:        23.7 MB

The following NEW packages will be INSTALLED:

  scipy              pkgs/main/osx-64::scipy-1.16.0-py311hcd1e4c7_0 

The following packages will be SUPERSEDED by a higher-priority channel:

  libgfortran        conda-forge::libgfortran-15.2.0-h7e5c~ --> pkgs/main::libgfortran-5.0.0-11_3_0_hecd8cb5_28 
  libgfortran5       conda-forge::libgfortran5-15.2.0-hd16~ --> pkgs/main::libgfortran5-11.3.0-h9dfd629_28 
  libopenblas  

### 2. Import libraries 

In [15]:
import pandas as pd
import numpy as np
import spacy
from spacy import displacy
import networkx as nx
import os
import matplotlib.pyplot as plt
import scipy
import re

In [20]:
print(spacy.__version__)

3.8.11


In [21]:
# Download English module

!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 7.7 MB/s  0:00:01 eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [22]:
# Load spacy English module

NER = spacy.load("en_core_web_sm")

### 3. Load text file 

In [23]:
# Load the 20th century textfile

with open('key_events_20th_century.txt', 'r', errors='ignore') as file:
    data = file.read().replace('\n', '')

In [24]:
# Create dataframe of list of countries

df_countries = pd.read_csv(('countries_list_20th_century_1.5.csv'), index_col = False)

### 4. Wrangling steps

The same wrangling steps from Ex 1.5 will need to be reapplied to make sure the list of countries can be searched for.  This is to replace entries such as Korea, North with North Korea. 

In [58]:
# Perform the same wrangling actions as in Ex 1.5:

# Remove spaces before and after country names

df_countries['country_name'] = df_countries['country_name'].str.strip()


In [59]:
# Some of the name are formatted in a way that won't allow them to be found in the text.
# Create a mapping of old names -> new names for the sake of counting them

replace_names = {
    'Korea, North': 'North Korea',
    'Korea, South': 'South Korea',
    "China, People's Republic of": 'China',
    'Bahamas, The': 'Bahamas',
    'Congo, Democratic Republic of the': 'Congo',
    'Congo, Republic of the': 'Congo',
    'Gambia, The':'Gambia',
    'Micronesia, Federated States of': 'Micronesia',
    'Vatican City (Holy See)': 'Vatican'}

# Apply the replacements
df_countries['country_name'] = df_countries['country_name'].replace(replace_names)


In [39]:
# Now add one space to the end of each country name.  This is for consistency, to stop words like 'Japanese' being counted as one mention 
# of Japan while the word 'French' would not count as one mention of 'France'.

# Delete this step now

#df_countries['country_name'] = df_countries['country_name'] + " "

In [60]:
df_countries.head(10)

,country_name
0,Afghanistan
1,Albania
2,Algeria
3,Andorra
4,Angola
5,Antigua and Barbuda
6,Argentina
7,Armenia
8,Australia
9,Austria


In [61]:
# Export df.countries to csv file

df_countries.to_csv('/Users/andymiller/20th-Century/list_of_countries_1.6.csv')

### 5. Create a NER object 

In [62]:
article = NER(data)

In [63]:
# Visualize identified entities

displacy.render(article[273:20000], style = "ent", jupyter = True)

### 6. Split sentence entities from NER object

In [64]:
df_sentences = [] # empty shell to store results

# Loop through sentences, get entity list for each sentence
for sent in article.sents:
    entity_list = [ent.text for ent in sent.ents]
    df_sentences.append({"sentence": sent, "entities": entity_list})
    
df_sentences = pd.DataFrame(df_sentences)

In [45]:
df_sentences.head(10)

,sentence,entities
0,"(Jump, to, contentMain, menuSearchDonateCreate...","[Jump, 20th, Historic, the 20th century, links..."
1,"(languagesArticleTalkReadEditView, historyTool...","[Wikipedia, encyclopediaThe 20th century]"
2,"(The, World, Wars, sparked, tension, between, ...","[The World Wars, the Cold War, the Space Race,..."
3,"(These, advancements, have, played, a, signifi...","[the 21st century, today]"
4,"(Historic, events, in, the, 20th, century[edit...","[the 20th, Edwardian, 1914The, the 20th century]"
5,"(The, 1900s, saw, the, decade, herald, a, seri...","[The 1900s, the decade]"
6,"(1914, saw, the, completion, of, the, Panama, ...","[1914, the Panama Canal]"
7,"(The, Scramble, for, Africa, continued, in, th...","[Scramble, Africa, the 1900s]"
8,"(The, atrocities, in, the, Congo, Free, State,...",[the Congo Free State]
9,"(From, 1914, to, 1918, ,, the, First, World, W...","[1914 to 1918, the First World War]"


### 7. Filter the entities according to country list 

In [53]:
df_countries.head()

,country_name
0,Afghanistan
1,Albania
2,Algeria
3,Andorra
4,Angola


In [65]:
# Function to filter out entities not of interest

def filter_entity(ent_list, df_countries):
    return [ent for ent in ent_list 
            if ent in list(df_countries['country_name'])]

In [67]:
# Check

filter_entity(["United States", "North Korea", "2", "banana"], df_countries)

['United States', 'North Korea']

In [68]:
df_sentences['country_entities'] = df_sentences['entities'].apply(lambda x: filter_entity(x, df_countries))

In [69]:
df_sentences['country_entities'].head(20)

0                    []
1                    []
2                    []
3                    []
4                    []
5                    []
6                    []
7                    []
8                    []
9                    []
10                   []
11                   []
12                   []
13     [France, Russia]
14    [Germany, Russia]
15            [Germany]
16            [Germany]
17                   []
18                   []
19                   []
Name: country_entities, dtype: object

In [70]:
# Filter out sentences that don't have any character entities

df_sentences_filtered = df_sentences[df_sentences['country_entities'].map(len) > 0]

In [71]:
df_sentences_filtered.head(20)

,sentence,entities,country_entities
13,"(After, a, period, of, diplomatic, and, milita...","[the July Crisis, the end of July 1914, the Br...","[France, Russia]"
14,"(The, Bolsheviks, negotiated, the, Treaty, of,...","[Bolsheviks, Germany, Russia]","[Germany, Russia]"
15,"(In, the, treaty, ,, Bolshevik, Russia, ceded,...","[Bolshevik Russia, Baltic, Germany, Kars Oblas...",[Germany]
16,"(It, also, recognized, the, independence, of, ...","[Germany, Allied, American]",[Germany]
21,"(Many, Germans, felt, these, reparations, were...","[Germans, Germany, Allied, Kaiser, Europe]",[Germany]
39,"(Germany, ,, 1933Fascism, first, appeared, in,...","[Germany, first, Italy, Benito Mussolini, 1922...","[Germany, Italy]"
40,"(The, ideology, was, supported, by, a, large, ...","[Adolf Hitler, Germany, 1933, Nazism, Germany,...","[Germany, Germany]"
41,"(The, Nazi, Party, in, Germany, was, dedicated...","[The Nazi Party, Germany, German, German, Cent...","[Germany, Germany]"
42,"(Antisemitism, during, the, Great, Depression,...","[the Great Depression, Jews, Austria, Austria,...","[Austria, Austria, Germany]"
45,"(Almost, all, of, the, new, democracies, in, t...","[Eastern Europe, Spain, Francisco Franco, the ...",[Spain]


### Create a Relationships Dataframe 

In [72]:


# window size = 5 : this defines how many sentences will be looked at simultaneously 
relationships = [] # create an empty list

for i in range(df_sentences_filtered.index[-1]):
    end_i = min(i+5, df_sentences_filtered.index[-1])
    country_list = sum((df_sentences_filtered.loc[i: end_i].country_entities), [])
    
    # Remove duplicated characters that are next to each other
    country_unique = [country_list[i] for i in range(len(country_list)) 
                   if (i==0) or country_list[i] != country_list[i-1]]
    
    if len(country_unique) > 1:
        for idx, a in enumerate(country_unique[:-1]):
            b = country_unique[idx + 1]
            relationships.append({"source": a, "target": b})

In [73]:
df_relationships = pd.DataFrame(relationships)

In [74]:
df_relationships

,source,target
0,France,Russia
1,France,Russia
2,Russia,Germany
3,Germany,Russia
4,France,Russia
...,...,...
651,India,Singapore
652,India,Singapore
653,India,Singapore
654,India,Singapore


In [75]:
# Sort the cases with a->b and b->a

df_relationships = pd.DataFrame(np.sort(df_relationships.values, axis = 1), columns = df_relationships.columns)
df_relationships.head(5)

,source,target
0,France,Russia
1,France,Russia
2,Germany,Russia
3,Germany,Russia
4,France,Russia


### 9. Export to csv file

In [77]:
# Export dataframe to csv file

df_relationships.to_csv('/Users/andymiller/20th-Century/country_relationships.csv')